# 26. Verificações de robustez do pareamento, do setor e da dependência espacial

Os resultados do [NB23](23_referencia_municipal_completa.ipynb), do [NB24](24_composicao_bases.ipynb) e do [NB25](25_setor_censitario.ipynb) dependem de escolhas que podem ser postas à prova com os próprios dados. Este notebook responde a quatro perguntas.

1. Quanto a escolha do ponto mais próximo, entre vários candidatos da mesma chave, reduz as distâncias medidas?
2. As discordâncias de setor resistem a uma tolerância de borda, variam com o nível da coordenada do CNEFE e são compatíveis com a malha usada?
3. O contraste entre as áreas de assentamento e o resto da cidade se mantém sem os registros próximos da divisa dessas áreas?
4. A autocorrelação espacial da cauda depende do mínimo de pares por setor, do limiar de 100 m ou dos apartamentos?

As tabelas saem em `outputs/referencia_completa/tables` com o prefixo `nb26_`.

In [1]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from libpysal.weights import Queen, Rook
from esda import Moran
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/config.py').exists())
sys.path.insert(0, str(ROOT))
from src import config
from src.metrics import chave_identidade, separar_logradouro_numero, conciliar_logradouros
from src.normalize import normalize_logradouro
from src.setores import carregar_setores_preliminares
P = config.PROCESSED_DIR
T = ROOT/'outputs/referencia_completa/tables'
SEED, PERM = config.SEED, 9999

C = gpd.read_parquet(P/'cnefe_bh.parquet')
B = gpd.read_parquet(P/'bhmap_bh.parquet', columns=['IDEND', 'SIGLA_TIPO', 'NOME_LOGRA', 'NUMERO_IMO', 'NOME_BAIRR', 'NOME_REGIO', 'CEP', 'geometry'])
I = pd.read_parquet(P/'cnefe_identidade_completa.parquet')
S = carregar_setores_preliminares()
vila = gpd.read_parquet(P/'vila_favela.parquet')
assert len(C) == 1180102 and len(B) == 757113 and I.index.equals(C.index)
em_vila = pd.Series(shapely.within(C.geometry.to_numpy(), vila.geometry.union_all()), index=C.index)
apto = pd.to_numeric(C.COD_TIPO_ESPECI, errors='coerce').eq(103).fillna(False).astype(bool)

def bairro(s):
    return s.map(lambda v: normalize_logradouro(v) if pd.notna(v) else pd.NA).astype('string[python]').replace('', pd.NA)
lc, nc = chave_identidade(*separar_logradouro_numero(C.LOGRAD_NUM))
lb, nb = chave_identidade(B.SIGLA_TIPO.fillna('')+' '+B.NOME_LOGRA.fillna(''), B.NUMERO_IMO)
mapa = conciliar_logradouros(lc.unique(), lb.unique(), 90, 2)
lc = lc.map(lambda x: mapa.get(x, x)).astype('string[python]')
nc, lb, nb = [s.astype('string[python]') for s in (nc, lb, nb)]
kc = (lc+'|'+nc).where(lc.notna() & lc.ne('') & nc.notna())
kb = (lb+'|'+nb).where(lb.notna() & lb.ne('') & nb.notna())
A = pd.DataFrame({'idx': C.index, 'chave': kc.to_numpy(), 'xc': C.geometry.x.to_numpy(), 'yc': C.geometry.y.to_numpy(),
                  'bairro_c': bairro(C.DSC_LOCALIDADE).to_numpy()})
R = pd.DataFrame({'idx_b': B.index, 'chave': kb.to_numpy(), 'xb': B.geometry.x.to_numpy(), 'yb': B.geometry.y.to_numpy(),
                  'bairro_b': bairro(B.NOME_BAIRR).to_numpy()})
J = A.dropna(subset=['chave']).merge(R.dropna(subset=['chave']), on='chave', how='inner')
J['dist'] = np.hypot(J.xc-J.xb, J.yc-J.yb)
base = J.loc[J.groupby('idx').dist.idxmin()].set_index('idx').sort_index()
assert len(base) == 1062044 and np.allclose(base.dist, I.dist_identidade.reindex(base.index), atol=1e-8)
print(f'Pares pela chave: {len(base):,} | candidatos avaliados: {len(J):,}')

Pares pela chave: 1,062,044 | candidatos avaliados: 1,485,150


## 1. Escolha do candidato mais próximo

Quando a chave tem vários pontos do BHMap, o par usa o mais próximo do registro do CNEFE. A regra favorece distâncias curtas. Para medir quanto, a tabela compara a distância ao candidato mais próximo com a distância ao mais distante, que é o limite oposto da escolha, e mede a amplitude entre os candidatos de cada registro.

In [2]:
g = J.groupby('idx').dist
coords = J.assign(xr=J.xb.round(2), yr=J.yb.round(2)).drop_duplicates(['idx', 'xr', 'yr']).groupby('idx').size()
cand = pd.DataFrame({'coordenadas': coords, 'mais_proximo': g.min(), 'mais_distante': g.max()})
cand['amplitude'] = cand.mais_distante - cand.mais_proximo
unico = cand.coordenadas.eq(1)
assert int((~unico).sum()) == 243456
def resumo(d):
    return pd.Series({'registros': len(d), 'mediana_m': d.median(), 'p90_m': d.quantile(.9), 'acima_100m_pct': 100*d.gt(100).mean()})
selecao = pd.DataFrame({
    'um candidato': resumo(cand.mais_proximo[unico]),
    'vários candidatos, mais próximo': resumo(cand.mais_proximo[~unico]),
    'vários candidatos, mais distante': resumo(cand.mais_distante[~unico]),
    'todos, mais próximo (regra adotada)': resumo(cand.mais_proximo),
    'todos, mais distante (limite)': resumo(cand.mais_distante)}).T
selecao['registros'] = selecao.registros.astype(int)
amplitude = pd.Series({'registros_varios_candidatos': int((~unico).sum()),
                       'amplitude_ate_30m_pct': 100*cand.amplitude[~unico].le(30).mean(),
                       'amplitude_acima_100m_pct': 100*cand.amplitude[~unico].gt(100).mean()})
selecao.to_csv(T/'nb26_selecao_candidato.csv')
amplitude.to_csv(T/'nb26_amplitude_candidatos.csv')
print(selecao.round(2).to_string()); print(amplitude.round(2).to_string())

                                     registros  mediana_m    p90_m  acima_100m_pct
um candidato                            818588      10.54    38.40            2.80
vários candidatos, mais próximo         243456       7.73    24.24            2.82
vários candidatos, mais distante        243456      12.83  1004.31           13.31
todos, mais próximo (regra adotada)    1062044       9.84    35.59            2.80
todos, mais distante (limite)          1062044      11.08    46.47            5.21
registros_varios_candidatos    243456.00
amplitude_ate_30m_pct              86.08
amplitude_acima_100m_pct           11.59


A escolha do mais próximo encurta as distâncias, mas pouco muda o retrato geral. Nos 243.456 registros cuja chave tem pontos do BHMap em coordenadas diferentes, a mediana até o candidato mais próximo é de 7,7 m, e até o mais distante, de 12,8 m, e em 86,1% deles o candidato mais distante está no máximo 30 m mais longe que o mais próximo, de modo que a escolha quase não importa. Mesmo no limite oposto, tomando sempre o candidato mais distante, a mediana da cidade sobe de 9,84 m para 11,08 m e o P90 de 35,6 m para 46,5 m. O efeito aparece na cauda, que passa de 2,8% para 5,2% dos pares, porque 11,6% desses registros têm candidatos a mais de 100 m entre si, o padrão das chaves ambíguas.

## 2. Tolerância de borda, nível da coordenada e coerência da malha

As três comparações de setor do NB25 são refeitas com tolerâncias de 5 a 50 m: a coordenada do CNEFE fora do próprio setor, o ponto do BHMap fora do setor declarado pelo CNEFE e o ponto do BHMap fora do setor que contém a coordenada do CNEFE. Com tolerância, um ponto só conta como fora se estiver a mais que essa distância do polígono. A segunda tabela separa as discordâncias e as distâncias dos pares pelo nível da coordenada do CNEFE. A terceira verifica se os códigos e as geometrias da malha preliminar correspondem: uma troca de versão da malha deixaria registros longe do setor declarado.

In [3]:
proprio = S.geometry.reindex(C.COD_SETOR).to_numpy()
dist_c = shapely.distance(proprio, C.geometry.to_numpy())
base['setor_c'] = C.COD_SETOR.reindex(base.index).to_numpy()
geom_decl = S.geometry.reindex(base.setor_c).to_numpy()
pts_b = B.geometry.reindex(base.idx_b).to_numpy()
dist_b = shapely.distance(geom_decl, pts_b)
jc = gpd.sjoin(C.loc[base.index, ['geometry']], S.reset_index()[['CD_SETOR', 'geometry']], how='left', predicate='within')
setor_pos = jc[~jc.index.duplicated()].CD_SETOR.reindex(base.index)
tem_pos = setor_pos.notna().to_numpy()
dist_pos = np.full(len(base), np.nan)
dist_pos[tem_pos] = shapely.distance(S.geometry.reindex(setor_pos[tem_pos]).to_numpy(), pts_b[tem_pos])
linhas = []
for tol in [0, 5, 10, 20, 50]:
    linhas.append({'tolerancia_m': tol,
                   'cnefe_fora_do_proprio_setor_pct': 100*np.mean(dist_c > tol),
                   'bhmap_fora_do_setor_declarado_pct': 100*np.mean(dist_b > tol),
                   'bhmap_fora_do_setor_da_coordenada_cnefe_pct': 100*np.nanmean(np.where(tem_pos, dist_pos > tol, np.nan))})
tolerancia = pd.DataFrame(linhas).set_index('tolerancia_m')
nivel_c = C.NV_GEO_COORD.astype(str)
por_nivel = pd.DataFrame({
    'registros': nivel_c.value_counts(),
    'cnefe_fora_10m_pct': pd.Series(dist_c > 10, index=C.index).groupby(nivel_c).mean()*100,
    'pares': nivel_c.reindex(base.index).value_counts(),
    'bhmap_fora_decl_10m_pct': pd.Series(dist_b > 10, index=base.index).groupby(nivel_c.reindex(base.index)).mean()*100,
    'mediana_par_m': base.dist.groupby(nivel_c.reindex(base.index)).median(),
    'p90_par_m': base.dist.groupby(nivel_c.reindex(base.index)).quantile(.9),
    'acima_100m_pct': base.dist.gt(100).groupby(nivel_c.reindex(base.index)).mean()*100}).sort_index()
codigo = pd.Series({'registros': len(C), 'codigos_na_malha_pct': 100*C.COD_SETOR.isin(S.index).mean(),
                    'a_mais_de_100m_do_setor': int((dist_c > 100).sum()), 'a_mais_de_1km_do_setor': int((dist_c > 1000).sum())})
tolerancia.to_csv(T/'nb26_setor_tolerancia.csv'); por_nivel.to_csv(T/'nb26_setor_por_nivel.csv'); codigo.to_csv(T/'nb26_setor_codigo_geometria.csv')
print(tolerancia.round(2).to_string()); print(por_nivel.round(2).to_string()); print(codigo.to_string())

              cnefe_fora_do_proprio_setor_pct  bhmap_fora_do_setor_declarado_pct  bhmap_fora_do_setor_da_coordenada_cnefe_pct
tolerancia_m                                                                                                                 
0                                       19.37                               2.27                                        20.86
5                                        8.70                               1.93                                        16.67
10                                       4.33                               1.69                                         5.06
20                                       1.59                               1.47                                         2.38
50                                       0.45                               1.15                                         1.51
              registros  cnefe_fora_10m_pct   pares  bhmap_fora_decl_10m_pct  mediana_par_m  p90_par_m  acima_100m_pct

A tolerância de borda separa dois comportamentos. As discordâncias que envolvem a coordenada do CNEFE caem rápido: a coordenada fora do próprio setor passa de 19,4% para 4,3% com 10 m e para 1,6% com 20 m, e o ponto do BHMap fora do setor da coordenada do CNEFE passa de 20,9% para 5,1% com 10 m. São pontos de entrada colados ao limite do setor. O ponto do BHMap fora do setor declarado, ao contrário, muda pouco, de 2,3% para 1,7% com 10 m e 1,5% com 20 m, o que indica discordância territorial de fato, e não efeito de borda.

Por nível da coordenada, as coordenadas estimadas (nível 3) ficam mais vezes a mais de 10 m do próprio setor (13,4%) e têm a maior cauda (9,4%), enquanto as de face de quadra (nível 4) quase nunca saem do setor, mas ficam mais longe do ponto municipal (mediana de 26,2 m). As coordenadas coletadas em campo (nível 1) têm mediana de 9,4 m, e as medianas de apartamentos (nível 2), de 11,9 m. Todos os códigos do CNEFE existem na malha preliminar, e nenhum registro fica a mais de 1 km do setor declarado, o que é coerente com códigos e malha da mesma versão.

## 3. Áreas de assentamento sem os registros próximos da divisa

A classificação dentro ou fora das áreas de assentamento usa a coordenada do CNEFE, e um registro perto da divisa pode cair do lado errado. A tabela refaz a cobertura pela chave, a cobertura com bairro e a cauda depois de excluir os registros a até 20 m e a até 50 m da divisa dessas áreas.

In [4]:
borda = vila.geometry.union_all().boundary
pts_c = C.geometry.to_numpy()
m_bairro = (J.bairro_c.notna() & J.bairro_b.notna() & J.bairro_c.eq(J.bairro_b)).fillna(False)
cb = J.loc[m_bairro]
com_bairro = cb.loc[cb.groupby('idx').dist.idxmin()].set_index('idx')
assert len(com_bairro) == 972070
par = I.pareado_identidade
linhas = []
for faixa in [0, 20, 50]:
    if faixa:
        zona = borda.buffer(faixa); shapely.prepare(zona)
        perto = pd.Series(shapely.intersects(zona, pts_c), index=C.index)
    else:
        perto = pd.Series(False, index=C.index)
    for nome, m in [('dentro', em_vila & ~perto), ('fora', ~em_vila & ~perto)]:
        ids = C.index[m]
        d = base.dist.reindex(ids).dropna()
        linhas.append({'faixa_excluida_m': faixa, 'territorio': nome, 'registros': len(ids),
                       'cobertura_chave_pct': 100*par[m].mean(), 'cobertura_com_bairro_pct': 100*C.index[m].isin(com_bairro.index).mean(),
                       'acima_100m_pct': 100*d.gt(100).mean()})
vila_borda = pd.DataFrame(linhas)
vila_borda.to_csv(T/'nb26_vila_sem_borda.csv', index=False)
print(vila_borda.round(2).to_string(index=False))

 faixa_excluida_m territorio  registros  cobertura_chave_pct  cobertura_com_bairro_pct  acima_100m_pct
                0     dentro     128149                72.33                     52.39            6.45
                0       fora    1051953                92.15                     86.02            2.46
               20     dentro      97728                73.21                     55.22            6.24
               20       fora    1017690                92.55                     86.96            2.42
               50     dentro      61092                74.11                     58.18            6.15
               50       fora     989076                92.70                     87.21            2.42


O contraste não depende dos registros da divisa. Sem os registros a até 50 m dela, a cobertura pela chave fica em 74,1% dentro das áreas de assentamento contra 92,7% fora, a cobertura com bairro em 58,2% contra 87,2% e a cauda em 6,2% contra 2,4%. Os valores com todos os registros eram 72,3% contra 92,1%, 52,4% contra 86,0% e 6,5% contra 2,5%.

## 4. Sensibilidade do I de Moran

O I de Moran da proporção da cauda, com a matriz Queen, é recalculado com mínimos de 10, 30 e 50 pares por setor e limiares de 50, 100 e 200 m para definir a cauda. Uma última linha retira os registros de apartamentos, que concentram as chaves com vários registros. Cada linha informa quantos setores e qual fração dos pares ficam fora do mínimo.

In [5]:
base['setor_c'] = C.COD_SETOR.reindex(base.index).to_numpy()
base['apto'] = apto.reindex(base.index).to_numpy()
def moran_cauda(d, minimo, limiar):
    a = d.groupby('setor_c').agg(pares=('dist', 'size'), cauda=('dist', lambda v: v.gt(limiar).mean()))
    geo = S.join(a, how='inner')
    elegiveis = geo[geo.pares >= minimo].reset_index()
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        q0, r0 = Queen.from_dataframe(elegiveis, use_index=False), Rook.from_dataframe(elegiveis, use_index=False)
    ilhas = sorted(set(q0.islands) | set(r0.islands))
    geo_m = elegiveis.drop(index=ilhas).reset_index(drop=True)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        w = Queen.from_dataframe(geo_m, use_index=False)
    w.transform = 'r'
    np.random.seed(SEED)
    m = Moran(geo_m.cauda.to_numpy(), w, permutations=PERM)
    return {'setores_fora_do_minimo': int(len(geo) - len(elegiveis)), 'pares_fora_do_minimo_pct': 100*geo.pares[geo.pares < minimo].sum()/len(d),
            'setores': len(geo_m), 'I': m.I, 'z': m.z_sim, 'p_unilateral': m.p_sim}
linhas = []
for minimo in [10, 30, 50]:
    for limiar in [50, 100, 200]:
        linhas.append({'recorte': 'todos os pares', 'minimo_pares': minimo, 'limiar_m': limiar, **moran_cauda(base, minimo, limiar)})
linhas.append({'recorte': 'sem apartamentos', 'minimo_pares': 30, 'limiar_m': 100, **moran_cauda(base[~base.apto], 30, 100)})
moran_sens = pd.DataFrame(linhas)
moran_sens.to_csv(T/'nb26_moran_sensibilidade.csv', index=False)
print(moran_sens.round(4).to_string(index=False))

         recorte  minimo_pares  limiar_m  setores_fora_do_minimo  pares_fora_do_minimo_pct  setores      I       z  p_unilateral
  todos os pares            10        50                      60                    0.0228     5037 0.2617 31.5758        0.0001
  todos os pares            10       100                      60                    0.0228     5037 0.2337 28.3560        0.0001
  todos os pares            10       200                      60                    0.0228     5037 0.1987 23.8306        0.0001
  todos os pares            30        50                     201                    0.2872     4896 0.2639 31.3089        0.0001
  todos os pares            30       100                     201                    0.2872     4896 0.2422 28.3582        0.0001
  todos os pares            30       200                     201                    0.2872     4896 0.2245 26.7615        0.0001
  todos os pares            50        50                     385                    0.9572     47

A autocorrelação positiva da cauda se mantém em todas as combinações. Com Queen, o I fica entre 0,19 e 0,26, sempre com p unilateral de 0,0001, o mínimo possível com 9.999 permutações. O mínimo de 30 pares deixa fora 201 setores com 0,3% dos pares, e o de 50, 385 setores com 1,0%. O I diminui com limiares maiores, porque a cauda fica mais rara, e não depende dos apartamentos: sem eles, é 0,26.

## Conclusão

Os resultados principais resistem às verificações. A escolha do candidato mais próximo encurta a cauda, mas não muda a mediana de forma relevante. As discordâncias da coordenada do CNEFE com o setor são quase todas de borda, enquanto a do ponto municipal com o setor declarado persiste com tolerância. O contraste das áreas de assentamento não vem dos registros próximos da divisa, e a autocorrelação da cauda não depende do mínimo de pares, do limiar de 100 m nem dos apartamentos.